<a href="https://colab.research.google.com/github/pendulumswings/pendulumswings.github.io/blob/main/scripts/chapter4/period_cromer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Period of simple pendulum computed by Cromer’s method and compared to Exact and small angle period.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk

def solve_pendulum_cromer(length, theta0_rad, total_time, dt, g=9.81):
    """
    Simulates the non-linear pendulum using the semi-implicit Euler-Cromer method.
    """

    # 1. Initialization
    num_steps = int(np.ceil(total_time / dt))
    t = np.linspace(0, (num_steps-1)*dt, num_steps)

    # Create arrays to store the history of the system
    theta = np.zeros(num_steps)
    omega = np.zeros(num_steps)

    theta[0] = theta0_rad
    omega[0] = 0.0

    # Pre-calculate constants
    alpha_coeff = -g / length

    # 2. Euler-Cromer Integration Loop
    for i in range(num_steps - 1):
        # Calculate angular acceleration based on CURRENT position
        accel = alpha_coeff * np.sin(theta[i])

        # UPDATE STEP 1: Update Omega FIRST
        omega[i+1] = omega[i] + accel * dt

        # UPDATE STEP 2: Update Theta SECOND (using NEW omega)
        theta[i+1] = theta[i] + omega[i+1] * dt

    # 3. Energy Calculation
    kinetic = 0.5 * (length**2) * omega**2
    potential = g * length * (1 - np.cos(theta))
    total_energy = kinetic + potential

    # 4. Period Extraction via Zero-Crossing Interpolation
    # np.where returns a tuple, take  to get the array of indices
    crossings_indices = np.where(np.diff(np.sign(theta)))[0]
    crossing_times = []

    for idx in crossings_indices:
        theta_prev = theta[idx]
        theta_next = theta[idx+1]
        t_prev = t[idx]

        # Check to avoid division by zero
        if np.abs(theta_next - theta_prev) > 1e-12:
            # Linear Interpolation
            fraction = np.abs(theta_prev / (theta_next - theta_prev))
            t_cross = t_prev + fraction * dt
            crossing_times.append(t_cross)

    crossing_times = np.array(crossing_times)

    # Calculate Period
    if len(crossing_times) >= 3:
        # Differences between alternating crossings (e.g. left-to-right to left-to-right)
        periods = crossing_times[2:] - crossing_times[:-2]
        T_numerical = np.mean(periods)
    else:
        T_numerical = np.nan
        print("Warning: Not enough zero crossings to estimate period.")

    # 5. Theoretical Calculations
    T_small = 2 * np.pi * np.sqrt(length / g)

    # Exact Period using Elliptic Integral (parameter m = k^2)
    k_squared = np.sin(theta0_rad / 2.0)**2
    K_val = ellipk(k_squared)
    T_exact = 4 * np.sqrt(length / g) * K_val

    if not np.isnan(T_numerical):
        error_rel = np.abs(T_numerical - T_exact) / T_exact
    else:
        error_rel = np.nan

    # return {
    #     "time": t,
    #     "theta": theta,
    #     "omega": omega,
    #     "energy": total_energy,
    #     "period_numerical": T_numerical,
    #     "period_exact": T_exact,
    #     "period_small_angle": T_small,
    #     "error_rel": error_rel,
    #     "params": {"L": length, "theta0": theta0_rad, "dt": dt}
    # }
    return T_numerical

# --- Usage Example ---
def main():
    # Parameters
    L = 1.0           # meters
    g = 9.81          # m/s^2
    dt = 0.0001        # Time step (s)

    # Angles from 0 to 180 in steps of 5
    angles_deg = np.arange(0, 181, 5)

    # Header
    print(f"{'Angle':<8} {'Linear(s)':<10} {'Exact(s)':<10} {'Cromer(s)':<10} {'Err vs Exact(%)':<18} {'Dev vs Linear(%)':<18}")
    print("-" * 80)

    for deg in angles_deg:
        rad = np.radians(deg)

        # 1. Theoretical Values
        # Small angle approximation (constant for all angles)
        T_small = 2 * np.pi * np.sqrt(L / g)

        # Exact Elliptic Integral
        if deg == 180:
            T_exact = np.inf
        else:
            k_squared = np.sin(rad / 2.0)**2
            T_exact = 4 * np.sqrt(L / g) * ellipk(k_squared)

        # 2. Run Simulation
        if deg == 0:
            # Pendulum doesn't move
            T_num = T_small # Limit case
        elif deg == 180:
            # Pendulum balanced at top
            T_num = np.inf
        else:
            # DYNAMIC TIME SCALING
            # We need the simulation to run long enough to see ~5 oscillations.
            # For large angles, T_exact becomes huge.
            estimated_period = T_exact
            T_sim = 5 * estimated_period

            # Run solver
            T_num = solve_pendulum_cromer(L, rad, T_sim, dt, g)

        # --- Formatting Output ---
        # 1. Handle Infinite Cases (180 degrees)
        if np.isinf(T_exact):
            exact_str = "inf"
            num_str = "inf"
            err_exact_str = "N/A"
            dev_lin_str = "inf"

        # 2. Handle Normal Cases
        else:
            exact_str = f"{T_exact:.6f}"

            if np.isnan(T_num):
                num_str = "No Data"
                err_exact_str = "N/A"
                dev_lin_str = "N/A"
            else:
                num_str = f"{T_num:.6f}"

                # Error: How far is simulation from the Exact Math?
                err_exact = abs((T_num - T_exact) / T_exact) * 100
                err_exact_str = f"{err_exact:.6f}"

                # Deviation: How far is simulation from Small Angle Approx?
                dev_lin = (T_num - T_small) / T_small * 100
                dev_lin_str = f"{dev_lin:.6f}"

        print(f"{deg:<8} {T_small:<10.4f} {exact_str:<10} {num_str:<10} {err_exact_str:<18} {dev_lin_str:<18}")
if __name__ == "__main__":
    main()

Angle    Linear(s)  Exact(s)   Cromer(s)  Err vs Exact(%)    Dev vs Linear(%)  
--------------------------------------------------------------------------------
0        2.0061     2.006067   2.006067   0.000000           0.000000          
5        2.0061     2.007022   2.007022   0.000000           0.047617          
10       2.0061     2.009893   2.009893   0.000000           0.190718          
15       2.0061     2.014694   2.014694   0.000000           0.430058          
20       2.0061     2.021451   2.021451   0.000000           0.766902          
25       2.0061     2.030201   2.030201   0.000000           1.203055          
30       2.0061     2.040990   2.040990   0.000000           1.740879          
35       2.0061     2.053878   2.053878   0.000000           2.383341          
40       2.0061     2.068938   2.068938   0.000000           3.134052          
45       2.0061     2.086256   2.086256   0.000000           3.997334          
50       2.0061     2.105935   2.105935